<style>
.reveal { font-family: 'Segoe UI', system-ui, sans-serif; }
.reveal h2 { color: #0D2240; border-bottom: 2.5px solid #1A7A9A;
             padding-bottom:.15em; margin-bottom:.5em; }
.reveal .slides section { text-align: left; }
.reveal pre { font-size:.62em; border-radius:6px; border:1px solid #E2E6EE; box-shadow:none; }
.reveal pre code { max-height:420px; }
.reveal ul li, .reveal ol li { margin-bottom:.3em; }
.defn { background:#EAF6FA; border-left:4px solid #1A7A9A;
        padding:.6em 1em; border-radius:0 6px 6px 0; margin:.5em 0; }
.nota { background:#FFF8E1; border-left:4px solid #C8961E;
        padding:.6em 1em; border-radius:0 6px 6px 0; margin:.5em 0; }
</style>

## Simulación Monte Carlo
### Modelado de Sistemas bajo Incertidumbre · Semanas 1-2
---
Departamento de Ingeniería Industrial · Universidad de los Andes

## Agenda
1. La idea fundamental del método Monte Carlo
2. Algoritmo básico: cuatro pasos
3. Distribuciones de probabilidad clave
4. Ejemplo 1 — Estimación de π
5. Ejemplo 2 — Dimensionamiento de flota
6. Análisis de convergencia
7. Métricas de riesgo: VaR y CVaR

## La Idea Fundamental
<div class="defn">

La **Simulación Monte Carlo** estima el comportamiento de un sistema generando muchos escenarios aleatorios y promediando sus resultados.

</div>

**Fundamento matemático:** Ley de los Grandes Números
$$\bar{X}_n = \frac{1}{n}\sum_{i=1}^n X_i \xrightarrow{n\to\infty} \mathbb{E}[X]$$

Mientras más réplicas, más precisa la estimación.

## Algoritmo Monte Carlo: 4 Pasos
1. **Modelar** las entradas aleatorias como distribuciones de probabilidad
2. **Generar** $n$ muestras aleatorias de esas distribuciones
3. **Evaluar** la función del sistema para cada muestra
4. **Analizar** la distribución de resultados

```python
# Estructura general
resultados = []
for _ in range(n_replicas):
    entrada = generar_muestra()       # paso 2
    salida  = evaluar_sistema(entrada) # paso 3
    resultados.append(salida)
analizar(resultados)                  # paso 4
```

## Distribuciones de Probabilidad Clave
| Distribución | Parámetros | Uso típico |
|---|---|---|
| **Uniforme** $U(a,b)$ | min, max | Incertidumbre sin información adicional |
| **Normal** $N(\mu,\sigma^2)$ | media, desv. est. | Errores de medición, procesos estables |
| **Exponencial** $\text{Exp}(\lambda)$ | tasa | Tiempos entre llegadas (Poisson) |
| **Triangular** $T(a,m,b)$ | min, moda, max | Duraciones de actividades |
| **Poisson** $\text{Pois}(\lambda)$ | tasa | Conteo de eventos en un período |

In [ ]:
import numpy as np, matplotlib.pyplot as plt
np.random.seed(1)
fig, axes = plt.subplots(1, 3, figsize=(10, 3))
dists = [
    (np.random.uniform(10, 30, 5000),   'Uniforme U(10,30)', '#1565C0'),
    (np.random.normal(20, 3, 5000),      'Normal N(20,3)',    '#1A7A9A'),
    (np.random.exponential(5, 5000),     'Exponencial(1/5)', '#6A1B9A'),
]
for ax, (data, title, color) in zip(axes, dists):
    ax.hist(data, bins=40, color=color, edgecolor='white', alpha=0.85)
    ax.set_title(title, fontsize=9); ax.set_yticks([])
plt.tight_layout(); plt.show()

In [ ]:
# Ejemplo 1: Estimación de π por Monte Carlo
import numpy as np, matplotlib.pyplot as plt
np.random.seed(42)

n = 50000
x, y = np.random.uniform(-1, 1, n), np.random.uniform(-1, 1, n)
dentro = x**2 + y**2 <= 1
pi_est = 4 * np.mean(dentro)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
# Scatter
idx = np.random.choice(n, 2000, replace=False)
axes[0].scatter(x[idx][dentro[idx]], y[idx][dentro[idx]], s=1, c='#1A7A9A', alpha=.5)
axes[0].scatter(x[idx][~dentro[idx]], y[idx][~dentro[idx]], s=1, c='#C8961E', alpha=.5)
axes[0].set_aspect('equal'); axes[0].set_title(f'π ≈ {pi_est:.5f}  (real: 3.14159...)')
# Convergencia
ns = np.logspace(1, np.log10(n), 200).astype(int)
pi_vals = [4*np.mean(dentro[:k]) for k in ns]
axes[1].semilogx(ns, pi_vals, '#0D2240', lw=1.5)
axes[1].axhline(np.pi, color='#C8961E', ls='--', lw=1.5, label='π real')
axes[1].set_xlabel('Número de muestras'); axes[1].set_ylabel('Estimación de π')
axes[1].legend(); axes[1].set_title('Convergencia')
plt.tight_layout(); plt.show()

In [ ]:
# Ejemplo 2: Dimensionamiento de flota logística
import numpy as np, matplotlib.pyplot as plt
np.random.seed(7)

# Demanda diaria de viajes: Normal(50, 8); capacidad por camión: Uniforme(8, 12) viajes/día
n = 10000
n_camiones_opciones = range(4, 9)
resultados = {}

for k in n_camiones_opciones:
    demanda     = np.random.normal(50, 8, n)
    cap_total   = k * np.random.uniform(8, 12, n)
    deficit     = np.maximum(demanda - cap_total, 0)
    costo_flota = k * 800_000                            # $800k/camión/mes
    costo_subcontrato = deficit * 150_000                 # $150k/viaje subcontratado
    costo_total = costo_flota + costo_subcontrato
    resultados[k] = costo_total

fig, ax = plt.subplots(figsize=(8, 3))
for k, costos in resultados.items():
    ax.plot([], [], lw=2, label=f'{k} camiones  E[C]=${np.mean(costos)/1e6:.1f}M')
labels = [f'{k} cam.' for k in n_camiones_opciones]
ax.boxplot([resultados[k]/1e6 for k in n_camiones_opciones],
           labels=labels, patch_artist=True,
           boxprops=dict(facecolor='#EAF6FA', color='#1A7A9A'),
           medianprops=dict(color='#C8961E', lw=2))
ax.set_ylabel('Costo total mensual (M$)'); ax.set_title('Costo por tamaño de flota')
plt.tight_layout(); plt.show()

## Métricas de Riesgo: VaR y CVaR
<div class="defn">

**Valor en Riesgo (VaR)** al nivel $\alpha$: el valor $v$ tal que la pérdida excede $v$ con probabilidad $1-\alpha$.

$$\text{VaR}_\alpha = F^{-1}(\alpha) = \inf\{x : P(X \le x) \ge \alpha\}$$

**CVaR** (Expected Shortfall): promedio de pérdidas que exceden el VaR.

$$\text{CVaR}_\alpha = \mathbb{E}[X \mid X > \text{VaR}_\alpha]$$
</div>

In [ ]:
import numpy as np, matplotlib.pyplot as plt
np.random.seed(3)
costos = np.random.gamma(shape=3, scale=500_000, size=10000)  # ejemplo
alpha = 0.95
var95  = np.percentile(costos, alpha*100)
cvar95 = costos[costos > var95].mean()

fig, ax = plt.subplots(figsize=(8, 3))
ax.hist(costos/1e6, bins=60, color='#1A7A9A', edgecolor='white', alpha=0.8, density=True)
ax.axvline(var95/1e6,  color='#C8961E', lw=2, ls='--', label=f'VaR 95% = ${var95/1e6:.2f}M')
ax.axvline(cvar95/1e6, color='#6A1B9A', lw=2, ls='-',  label=f'CVaR 95% = ${cvar95/1e6:.2f}M')
ax.set_xlabel('Costo (M$)'); ax.set_ylabel('Densidad')
ax.set_title('Distribución de costos y métricas de riesgo'); ax.legend()
plt.tight_layout(); plt.show()

## Conclusiones
- Monte Carlo convierte **incertidumbre en distribuciones de resultados**
- Funciona para cualquier sistema modelable en código
- La convergencia mejora con $\sqrt{n}$ — duplicar precisión requiere 4× más muestras
- Las métricas de riesgo (VaR, CVaR) permiten tomar decisiones considerando el peor escenario
- Base para todos los temas siguientes del curso